In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

# --- CONFIGURAÇÕES DO SISTEMA ---
M = 2               
sps = 8             
num_symbols = 5000  
rolloff = 0.25      
noise_on = True     # Ativado para observar a interação Ruído vs ISI
noise_var = 0.01    # Variância fixa do ruído
filter_type = 'SRRC' 

# --- CONFIGURAÇÃO DO CANAL ---
R = 1.0             
fs = sps * R        
channel_cutoff = 0.6 * R # Reduza para induzir ISI (ex: 0.45)
channel_gain = 1.0      # <--- NOVO: Controle de atenuação/ganho para variar SNR

# --- 1. GERAÇÃO DE SÍMBOLOS E UPSAMPLING ---
symbols = np.random.choice(np.arange(-(M-1), M, 2), num_symbols)
upsampled = np.zeros(num_symbols * sps)
upsampled[::sps] = symbols

# --- 2. DEFINIÇÃO DO FILTRO (Apenas SRRC) ---
def get_filter(ftype, sps, beta):
    span = 64 # Aumentado para melhor precisão em roll-offs baixos
    t = np.arange(-span/2 * sps, span/2 * sps + 1) / sps
    h = np.zeros(len(t))
    b = max(beta, 1e-9) 
    for i, ti in enumerate(t):
        if ti == 0: 
            h[i] = 1 - b + 4*b/np.pi
        elif abs(ti) == 1/(4*b):
            h[i] = (b/np.sqrt(2)) * ((1+2/np.pi)*np.sin(np.pi/(4*b)) + (1-2/np.pi)*np.cos(np.pi/(4*b)))
        else:
            num = np.sin(np.pi*ti*(1-b)) + 4*b*ti*np.cos(np.pi*ti*(1+b))
            den = np.pi*ti*(1-(4*b*ti)**2)
            h[i] = num / den
    h *= np.hamming(len(t))
    return h / np.sqrt(np.sum(h**2))

tx_filter = get_filter(filter_type, sps, rolloff)
rx_filter = tx_filter 

# --- 3. PROCESSAMENTO COM CANAL (BANDA + ATENUAÇÃO) ---
tx_signal = np.convolve(upsampled, tx_filter, mode='same')

# Canal: Banda limitada + Ganho/Atenuação
nyq_fs = fs / 2
h_channel = signal.firwin(101, channel_cutoff / nyq_fs, window='hamming')
# Aplicamos o filtro de banda e o ganho do canal
chan_output = np.convolve(tx_signal, h_channel, mode='same') * channel_gain

# Adição de ruído (independente do ganho do sinal para variar SNR)
rx_input = chan_output + (np.random.normal(0, np.sqrt(noise_var), len(chan_output)) if noise_on else 0)
rx_output = np.convolve(rx_input, rx_filter, mode='same')

# Resposta Global P(t) (Normalizada para visualização da ISI pura)
p_t_mid = np.convolve(tx_filter, h_channel, mode='full')
p_t = np.convolve(p_t_mid, rx_filter, mode='full')
t_p = (np.arange(len(p_t)) - len(p_t)//2) / sps

# --- 4. VISUALIZAÇÃO ---
fig, axs = plt.subplots(3, 2, figsize=(15, 15)) 

# A. FORMA DE ONDA (Com indicação de ganho)
axs[0, 0].plot(rx_output[len(rx_output)//2 : len(rx_output)//2 + sps*40], color='tab:blue')
axs[0, 0].step(np.arange(0, sps*40, sps), symbols[num_symbols//2:num_symbols//2+40] * channel_gain, where='mid', color='tab:red', alpha=0.4, label='Nível Ideal (com Ganho)')
axs[0, 0].set_title(f'Sinal com Ganho={channel_gain} e Ruído={noise_var}')
axs[0, 0].grid(True, alpha=0.3)
axs[0, 0].legend()

# B. DIAGRAMA DE OLHO
# Aqui é onde você observará a lógica do slide
start, end = 100 * sps, len(rx_output) - 100 * sps
for i in range(start, end - 2*sps, 2*sps):
    axs[0, 1].plot(rx_output[i : i + 2*sps], color='black', alpha=0.07) 
axs[0, 1].set_title('Diagrama de Olho (Efeito Combinado Ruído + ISI)')
axs[0, 1].grid(True, alpha=0.2)

# C. RESPOSTA AO IMPULSO P(t) (ISI Pura)
axs[1, 0].plot(t_p, p_t / np.max(p_t), color='black', lw=1.5)
samples_pts = np.arange(-4, 5, 1)
axs[1, 0].plot(samples_pts, [1 if x==0 else 0 for x in samples_pts], 'ro', label='Instantes T')
axs[1, 0].set_title(f'P(t) Normalizado (Mostra apenas a ISI)')
axs[1, 0].set_xlim([-5, 5])
axs[1, 0].grid(True, alpha=0.3)

# D. RESPOSTA EM FREQUÊNCIA P(f)
N_fft = 4096
P_f = np.fft.fftshift(np.fft.fft(p_t, N_fft))
freqs = np.linspace(-fs/2, fs/2, N_fft)
axs[1, 1].plot(freqs, 20 * np.log10(np.abs(P_f) / np.max(np.abs(P_f))), color='darkgreen')
axs[1, 1].axvline(channel_cutoff, color='red', linestyle='--', label='Corte Canal')
axs[1, 1].set_title('Espectro vs Corte do Canal')
axs[1, 1].set_xlim([-R, R]); axs[1, 1].set_ylim([-60, 5]); axs[1, 1].grid(True, alpha=0.3)

# E. ZOOM AMOSTRAGEM
num_view = 15 
start_idx = len(rx_output)//2
time_range = np.arange(0, num_view * sps)
signal_segment = rx_output[start_idx : start_idx + num_view * sps]
sample_idx = np.arange(0, num_view * sps, sps)
sample_vals = signal_segment[sample_idx]

axs[2, 0].plot(time_range, signal_segment, color='tab:blue')
axs[2, 0].plot(sample_idx, sample_vals, 'ro')
axs[2, 0].set_title('Instantes de Decisão')
axs[2, 0].grid(True, alpha=0.3)

# F. RESUMO E LÓGICA DO SLIDE
axs[2, 1].axis('off')
axs[2, 1].text(0.1, 0.2, (f"Lógica do Experimento:\n\n"
                          f"1. Mantenha 'noise_var' fixo.\n"
                          f"2. Aumente 'channel_gain' (Ex: de 0.5 para 2.0).\n"
                          f"3. Se o olho abrir: Degradação era Ruído.\n"
                          f"4. Se o olho NÃO abrir: Degradação é ISI.\n\n"
                          f"Config Atual:\n"
                          f"- SNR Relativa: Altura Sinal/Ruído\n"
                          f"- Banda Canal: {channel_cutoff} Hz\n"
                          f"- Ganho Canal: {channel_gain}"), 
               fontsize=11, family='monospace', bbox=dict(facecolor='wheat', alpha=0.3))

plt.tight_layout()
plt.show()